In [ ]:
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
fraud_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(fraud_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_value = (df.isnull().sum() / len(df))


missing_data = pd.DataFrame({
    'Column': missing_value.index,
    'Missing_Percentage': missing_value.values
})

missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)
print("Missing Data Analysis:")
missing_data.head(10)

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")


# handle them
for col in missing_data.Column:
  df[col].fillna(df[col].median(), inplace=True)

check_missing_values(df)


In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

# if there any I just write it , for you to know I  understand , Iknow there is non catagoris
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True) * 100)
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()
check_target_imbalance(df, "Target")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:

!pip install catboost

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# Task 2,3,4,5: Write your code here:

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold

lr_accuracy = []
lr_f1 = []

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models


  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  lr_f1.append(f1)
  lr_accuracy.append(accuracy)

lr_accuracy = np.array(lr_accuracy)
print(f"\nMAE:  ${lr_accuracy.mean():,.2f}")
lr_f1 = np.array(lr_f1)
print(f"\nMAE:  ${lr_f1.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Plot feature importance
feature = X.columns
importance = pd.DataFrame({
    'feature': feature,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# from plot above we notice that  P_2  is the most important
print("The most imortant feature , the golde one :\n(P_2)")

In [ ]:
# Task Bonus: Write your code here:
X = df["P_2"]

# Task 2,3,4,5: Write your code here:

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold

lr_accuracy_golden = []
lr_f1_golden = []

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models


  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  y_pred.numpy()

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  lr_f1.append(f1)
  lr_accuracy.append(accuracy)



lr_accuracy_golden = np.array(lr_accuracy_golden)
print("Model with golden feature : \n")
print(f"\nMAE:  ${lr_accuracy_golden.mean():,.2f}")
lr_f1_golden = np.array(lr_f1_golden)
print(f"\nMAE:  ${lr_f1_golden.mean():,.2f}")



print("\n\nModel with all feature : \n")
print(f"\nMAE:  ${lr_accuracy.mean():,.2f}")
lr_f1 = np.array(lr_f1)
print(f"\nMAE:  ${lr_f1.mean():,.2f}")


